# BISINDO YOLO26 — kandidat frame + review gesture
RQ dan training tetap: COCO → 6 kelas dan COCO → 8 kelas, dua training independen dari checkpoint identik.

**Satu positive frame per video hanya setelah reviewer memastikan gesture sesuai kelas.** Nama video dan keberhasilan MediaPipe bukan bukti gesture benar. Tidak ada label positif otomatis untuk preparation/resting/transition pose.

Pilih kernel **thesis_cuthbert**, restart, jalankan dari atas. Hasil baru di `experiment_yolo26_v3`; hasil v2 dipertahankan. Class mapping dan split dibekukan dari inventory v2, tidak dihitung ulang.

Dataset: saya, kamu, senang, bingung, marah, apa_kabar, halo, tidur. Video tanpa pilihan yang layak tetap `needs_review`, tidak dipaksa berlabel dan tidak diam-diam dibuang. Training dan evaluasi default OFF.

In [1]:
import importlib.metadata as package_metadata
required = {"ultralytics": "8.4.197", "mediapipe": "0.10.20", "protobuf": "4.25.8", "numpy": "1.26.4"}
problems = []
for package, version in required.items():
    try:
        installed = package_metadata.version(package)
        if installed != version:
            problems.append(f"{package}: {installed}, expected {version}")
    except package_metadata.PackageNotFoundError:
        problems.append(f"{package}: missing")
if problems:
    raise RuntimeError("Pilih kernel thesis_cuthbert. Environment mismatch:\n" + "\n".join(problems))

from pathlib import Path
import json, sys, time
import numpy as np
import pandas as pd
import cv2
import torch
from IPython.display import display, HTML, Image
import bisindo_pipeline as pipeline
from ultralytics import YOLO
import importlib
pipeline=importlib.reload(pipeline)
import frame_candidates as frame_pipeline
frame_pipeline=importlib.reload(frame_pipeline)

ROOT = Path.cwd().resolve()
if not (ROOT / 'bisindo_pipeline.py').is_file():
    raise RuntimeError('Jalankan notebook dari folder proyek')
WORK = ROOT / 'experiment_yolo26_v3'
WORK.mkdir(exist_ok=True)
BASE, NAMES = pipeline.BASE, pipeline.CLASSES
IMGSZ, CONF, MATCH_IOU = 640, 0.25, 0.50
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
SEEDS = [42]  # putuskan beberapa seed sebelum eksperimen jika diperlukan
RUN_TRAINING = False
RUN_EVALUATION = False
print('Kernel:', sys.executable)
print('Device:', DEVICE)

Kernel: C:\Users\Ricardo Cuthbert\Downloads\skrpsi bwabwa\.venv-bisindo\Scripts\python.exe
Device: 0


## 1. Inventory dan split yang sudah ditetapkan
`split_lock.csv` disalin sekali dari inventory v2. Path video, hash sumber, mapping kelas, source dan split wajib konsisten. Perubahan inventory ditolak dan harus direkonsiliasi eksplisit; pemilihan kandidat tidak pernah memindahkan video antar split.

In [2]:
inventory=frame_pipeline.freeze_inventory(ROOT,WORK)
display(inventory.groupby(['class_name','split']).size().unstack(fill_value=0).reindex(NAMES))
print('Frozen split:',WORK/'split_lock.csv')

Frozen split: C:\Users\Ricardo Cuthbert\Downloads\skrpsi bwabwa\experiment_yolo26_v3\split_lock.csv


split,test,train,val
class_name,,,
saya,50,240,60
kamu,50,240,60
senang,50,240,60
bingung,50,240,60
marah,50,240,60
apa_kabar,50,240,60
halo,50,240,60
tidur,50,240,60


## 2. Beberapa kandidat, bukan satu frame otomatis
Sampling deterministik pada **20%, 30%, 40%, 50%, 60%, 70%, 80%** video (indeks integer unik). MediaPipe IMAGE mode, maksimal dua tangan. Jika tidak ada yang lolos cek teknis, coba titik tambahan 5%,10%,15%,85%,90%,95%.

Cek: 21 landmark finite/hand, jarak landmark dari tepi ≥1%, sisi tangan ≥16 px, area ≥256 px, box valid, skor fokus minimum per ROI tangan pada resize 128×128. Laplacian variance <60 diberi flag `possible_hand_blur`. Threshold ini heuristik awal, belum terkalibrasi sebagai standar kualitas. Occlusion dan ketepatan gesture tidak bisa dibuktikan oleh landmark/fokus.

Skor hanya merekomendasikan kandidat teknis. Semua video tetap `needs_review` sampai manusia memilih frame yang merepresentasikan gesture. Tidak ada file label YOLO positif sebelum pilihan tersebut. Kandidat dan alasan flag tetap disimpan.

Sel ini bisa dilanjutkan ulang; cache per video terikat hash sumber, recipe/model, dan JPEG. Pemrosesan pertama lebih lama daripada ekstraksi satu frame.

In [3]:
annotations=pipeline.prepare(ROOT,WORK)
environment=pipeline.record_environment(WORK)
display(json.loads((WORK/'annotation_summary.json').read_text()))
display(pd.read_csv(WORK/'annotation_counts.csv'))
display(pd.read_csv(WORK/'flagged_candidates.csv').head(30))

Candidate videos: 50/2800
Candidate videos: 100/2800
Candidate videos: 150/2800
Candidate videos: 200/2800
Candidate videos: 250/2800
Candidate videos: 300/2800
Candidate videos: 350/2800
Candidate videos: 400/2800
Candidate videos: 450/2800
Candidate videos: 500/2800
Candidate videos: 550/2800
Candidate videos: 600/2800
Candidate videos: 650/2800
Candidate videos: 700/2800
Candidate videos: 750/2800
Candidate videos: 800/2800
Candidate videos: 850/2800
Candidate videos: 900/2800
Candidate videos: 950/2800
Candidate videos: 1000/2800
Candidate videos: 1050/2800
Candidate videos: 1100/2800
Candidate videos: 1150/2800
Candidate videos: 1200/2800
Candidate videos: 1250/2800
Candidate videos: 1300/2800
Candidate videos: 1350/2800
Candidate videos: 1400/2800
Candidate videos: 1450/2800
Candidate videos: 1500/2800
Candidate videos: 1550/2800
Candidate videos: 1600/2800
Candidate videos: 1650/2800
Candidate videos: 1700/2800
Candidate videos: 1750/2800
Candidate videos: 1800/2800
Candidate vi

{'videos': 2800, 'candidate_frames': 33250, 'technically_eligible_candidates': 1120, 'flagged_candidates': 32130, 'selected_positives': 0, 'needs_review': 2800, 'training_allowed': False, 'note': 'Eligible means technical checks passed, NOT that the gesture matches the class. No positives without human selection.'}

,class_name,source,split,frame_review_status,samples
0,apa_kabar,test,test,needs_review,50
1,apa_kabar,train,train,needs_review,240
2,apa_kabar,train,val,needs_review,60
3,bingung,test,test,needs_review,50
4,bingung,train,train,needs_review,240
5,bingung,train,val,needs_review,60
6,halo,test,test,needs_review,50
7,halo,train,train,needs_review,240
8,halo,train,val,needs_review,60
9,kamu,test,test,needs_review,50


,sample_id,candidate_id,frame_index,reason
0,546e41061b809d1b8dbb,546e41061b809d1b8dbb_f00000017,17,possible_hand_blur
1,546e41061b809d1b8dbb,546e41061b809d1b8dbb_f00000025,25,possible_hand_blur
2,546e41061b809d1b8dbb,546e41061b809d1b8dbb_f00000042,42,possible_hand_blur
3,546e41061b809d1b8dbb,546e41061b809d1b8dbb_f00000050,50,possible_hand_blur
4,546e41061b809d1b8dbb,546e41061b809d1b8dbb_f00000058,58,possible_hand_blur
5,546e41061b809d1b8dbb,546e41061b809d1b8dbb_f00000066,66,no_hands
6,12725eb6a63eaac28e82,12725eb6a63eaac28e82_f00000017,17,possible_hand_blur
7,12725eb6a63eaac28e82,12725eb6a63eaac28e82_f00000026,26,possible_hand_blur
8,12725eb6a63eaac28e82,12725eb6a63eaac28e82_f00000035,35,possible_hand_blur
9,12725eb6a63eaac28e82,12725eb6a63eaac28e82_f00000052,52,possible_hand_blur


## 3. Contact sheet dan pemilihan frame di HTML
Buka `frame_review.html` di Chrome/Edge. Setiap video memiliki contact sheet kandidat + bounding box + indeks + fokus + flag. Cari/filter per kelas/source/status. Klik video untuk membandingkan kandidat, atau putar video sumber untuk konteks gerakan.

- Pilih kandidat yang benar-benar menunjukkan gesture kelas. Pilih tangan relevan lewat checkbox atau drag kotak gabungan. Centang verifikasi gesture, lalu **Pilih frame gesture valid**.
- Preparation/resting/transition/blur/terpotong: **Tolak kandidat ini**, pilih alasannya, lalu pilih kandidat lain.
- Jika belum ada yang layak: isi alasan → **Cari frame lain dari video**. Unduh dan impor JSON; pipeline mencari kandidat tambahan untuk video tersebut.
- Jika tetap tidak ada: isi alasan → **needs_review**. Video tidak hilang dari manifest dan tidak menjadi sampel positif.
- Kandidat berflag teknis memerlukan justifikasi manusia jika tetap dipilih. Jangan override frame yang benar-benar blur/terpotong hanya agar gate lolos.

Pilihan massal tersedia hanya atas konfirmasi eksplisit reviewer setelah menilai gesture seluruh rekomendasi. Tidak dijalankan otomatis dan tidak menyetujui kandidat yang sudah ditolak. Unduh draft untuk cadangan progres.

In [4]:
frame_pipeline.render_report(WORK)
print('Contact sheets:',WORK/'contact_sheets')
print('Review:',WORK/'frame_review.html')
display(HTML('<a target="_blank" href="experiment_yolo26_v3/frame_review.html">Buka candidate frame review</a>'))

Contact sheets: C:\Users\Ricardo Cuthbert\Downloads\skrpsi bwabwa\experiment_yolo26_v3\contact_sheets
Review: C:\Users\Ricardo Cuthbert\Downloads\skrpsi bwabwa\experiment_yolo26_v3\frame_review.html


<IPython.core.display.HTML object>

## 4. Impor review dan cari frame lain
Ekspor HTML: `bisindo_frame_review.json`. Sel ini membaca folder Downloads; ubah path bila file disimpan di lokasi lain. Impor memeriksa snapshot, kandidat milik video, hash JPEG/sumber, mapping kelas, verifikasi gesture, box, dan alasan override/penolakan.

Pilihan diterapkan ke manifest `annotations.csv`: `frame_index`, `selected_candidate_id`, `frame_review_status`, `rejection_reason`, `rejected_candidates`, reviewer, hash. Hanya frame terpilih disalin ke `frames/` dan menghasilkan satu label YOLO. Manifest sebelum impor dicadangkan.

Permintaan frame lain diproses setelah impor. Setelah itu buka ulang HTML yang baru dibuat; jangan memakai snapshot lama. Data belum dipilih tetap needs_review, tanpa positive frame.

In [5]:
REVIEW_JSON=Path.home()/'Downloads'/'bisindo_frame_review.json'
if REVIEW_JSON.exists():
    try:
        annotations=frame_pipeline.import_review(ROOT,WORK,REVIEW_JSON)
        display(annotations.frame_review_status.value_counts().rename('videos'))
        print('Review diimpor. Buka ulang frame_review.html dan jalankan validation gates.')
    except ValueError as exc:
        print('Review tidak diterapkan:',exc)
else:
    print('Belum ada review frame. Ekspor dari HTML setelah QC:',REVIEW_JSON)

Belum ada review frame. Ekspor dari HTML setelah QC: C:\Users\Ricardo Cuthbert\Downloads\bisindo_frame_review.json


## 5. Validation gates sebelum training
Wajib satu frame gesture yang disetujui per video, indeks/candidate/JPEG sesuai, review gesture eksplisit, alasan override bila berflag, box valid, class mapping dan frozen split konsisten. Semua video harus tetap tercatat. Video needs_review menahan training; tidak otomatis dikeluarkan untuk memenuhi angka sampel.

Label dibangun dari tangan relevan pada JPEG terpilih. Enam kelas bersama tetap byte-identik pada dataset A/B. Label kelas Kaggle tetap dari ground truth CSV dan terikat frame yang dipilih.

In [6]:
try:
    checked=pipeline.validate_dataset(ROOT,WORK)
    print('QC lulus. Training tetap OFF sampai diaktifkan sengaja.')
except RuntimeError as exc:
    print(exc)
    report=json.loads((WORK/'validation_report.json').read_text())
    print('Contoh temuan:',*report['errors'][:10],sep='\n')

QC not passed: 8400 findings. See C:\Users\Ricardo Cuthbert\Downloads\skrpsi bwabwa\experiment_yolo26_v3\validation_report.json. Training blocked.
Contoh temuan:
546e41061b809d1b8dbb: no human-verified representative gesture frame
12725eb6a63eaac28e82: no human-verified representative gesture frame
d2633193e8fb2c60fe02: no human-verified representative gesture frame
83195ead14c77100688a: no human-verified representative gesture frame
33f389c544311ca29559: no human-verified representative gesture frame
2c4d6a235cc09aa1c3d3: no human-verified representative gesture frame
e9f79f363d6c34ff858d: no human-verified representative gesture frame
d168820cc1e98c8533ec: no human-verified representative gesture frame
5a12428a3344d508dbe8: no human-verified representative gesture frame
cef35cad731154154f69: no human-verified representative gesture frame


## 6. Checkpoint resmi dan training independen
Checkpoint YOLO26n resmi Hugging Face, revision/hash tetap. A dan B selalu memuat file COCO yang sama dari awal, bukan A → B. Versi efektif di environment.json / requirements-resolved.txt. Konfigurasi training tetap 640×640, batch16, 100 epoch, MuSGD, cosine LR, FP32 dan seed42. Step count tidak disamakan.

Pemilihan frame dan quality control adalah prasyarat dataset. Tidak ada training yang dijalankan untuk memilih kandidat.

In [7]:
checkpoint=pipeline.official_checkpoint(WORK)
initial=YOLO(str(checkpoint))
assert initial.task=='detect' and len(initial.names)==80 and initial.names[0]=='person'
display(json.loads((WORK/'checkpoint.json').read_text()))
del initial

{'repo_id': 'Ultralytics/YOLO26', 'filename': 'yolo26n.pt', 'revision': '070ac3c51435984d992ba92eb3f234daf4e4500d', 'path': 'C:\\Users\\Ricardo Cuthbert\\Downloads\\skrpsi bwabwa\\experiment_yolo26_v2\\hf_cache\\models--Ultralytics--YOLO26\\snapshots\\070ac3c51435984d992ba92eb3f234daf4e4500d\\yolo26n.pt', 'sha256': '9b09cc8bf347f0fc8a5f7657480587f25db09b34bf33b0652110fb03a8ad4fef'}

## 7. Training — default OFF
Gate dipanggil ulang di fungsi training. Jangan menyalakan sebelum seluruh video memiliki satu frame representatif yang telah disetujui.

In [8]:
if RUN_TRAINING:
    DATA,BEST=pipeline.train_independent(ROOT,WORK,seeds=SEEDS)
else:
    print('Training OFF; selesaikan pemilihan frame dan QC dahulu.')

Training OFF; selesaikan pemilihan frame dan QC dahulu.


## 8. Evaluasi setelah training — default OFF
Aktifkan `RUN_EVALUATION=True` setelah training tersedia. Utama: A/B pada 300 gambar enam kelas yang identik. Tambahan: B pada 400 gambar delapan kelas. Seluruh kelas output B tetap dipertahankan; tidak memakai filter `classes`.

mAP macro dihitung atas kelas dengan ground truth. Karena FP dua kelas tambahan tidak langsung masuk macro-average AP kelas lama, hitung juga precision/recall/F1 **mikro threshold tetap** dari semua output. `validator_precision/recall` memakai titik max-F1 validator, berbeda dari metrik threshold tetap. Simpan keduanya. Jangan memilih threshold memakai test.

In [9]:
if RUN_EVALUATION:
    pipeline.validate_dataset(ROOT, WORK)
    trained=json.loads((WORK/'trained_models.json').read_text())
    DATA=Path(trained['dataset'])
    if json.loads((DATA/'release.json').read_text())['annotations_sha256'] != pipeline.sha(WORK/'annotations.csv'):
        raise RuntimeError('Anotasi berubah sejak training; evaluasi ditahan')
    BEST={(r['model'],r['seed']):Path(r['path']) for r in trained['models']}
    df=pd.read_csv(DATA/'manifest.csv',keep_default_na=False)
    frames=WORK/'frames'
    sample_id=pipeline.sid
else:
    print('Evaluasi OFF.')

Evaluasi OFF.


In [10]:
if RUN_EVALUATION:
    summary, per_class = [], []
    for (name, seed), weights in BEST.items():
        for scope in (["common6", "full8"] if name == "B" else ["common6"]):
            config = DATA / name / ("full.yaml" if scope == "full8" else "common.yaml")
            result = YOLO(str(weights)).val(data=str(config), split="test", imgsz=IMGSZ,
                batch=1, device=DEVICE, workers=0, conf=0.001, iou=0.7,
                rect=False, quantize=32, plots=True, project=str(WORK / "evaluation"), name=f"{name}_{seed}_{scope}")
            summary.append(dict(model=name, seed=seed, scope=scope,
                mAP50=float(result.box.map50), mAP50_95=float(result.box.map),
                validator_precision=float(result.box.mp), validator_recall=float(result.box.mr)))
            for i, cls in enumerate(result.box.ap_class_index):
                per_class.append(dict(model=name, seed=seed, scope=scope, class_name=NAMES[int(cls)],
                    AP50=float(result.box.ap50[i]), AP50_95=float(result.box.ap[i]),
                    validator_precision=float(result.box.p[i]), validator_recall=float(result.box.r[i])))
    summary = pd.DataFrame(summary)
    summary.to_csv(WORK / "detection_summary.csv", index=False)
    pd.DataFrame(per_class).to_csv(WORK / "per_class.csv", index=False)
    display(summary)

In [11]:
def box_iou(box, gt):
    x1, y1 = np.maximum(box[:2], gt[:2])
    x2, y2 = np.minimum(box[2:], gt[2:])
    inter = max(0, x2-x1) * max(0, y2-y1)
    a = (box[2]-box[0]) * (box[3]-box[1])
    b = (gt[2]-gt[0]) * (gt[3]-gt[1])
    return inter / max(a+b-inter, 1e-12)

def score_one(boxes, pred_cls, confs, gt, gt_cls, threshold=MATCH_IOU):
    matched, tp, fp = False, 0, 0
    for i in np.argsort(-np.asarray(confs)):
        if not matched and int(pred_cls[i]) == gt_cls and box_iou(boxes[i], gt) >= threshold:
            tp += 1
            matched = True
        else:
            fp += 1
    return np.array([tp, fp, int(not matched)])

gt = np.array([0., 0., 10., 10.])
assert np.array_equal(score_one([gt], [0], [0.9], gt, 0), [1, 0, 0])
assert np.array_equal(score_one([gt], [6], [0.9], gt, 0), [0, 1, 1])
assert np.array_equal(score_one([gt, gt], [0, 0], [0.9, 0.8], gt, 0), [1, 1, 0])
assert np.array_equal(score_one([], [], [], gt, 0), [0, 0, 1])
assert np.array_equal(score_one([[20, 20, 30, 30]], [0], [0.9], gt, 0), [0, 1, 1])
print("Matching self-checks passed")

Matching self-checks passed


In [12]:
if RUN_EVALUATION:
    fixed, timing, predictions = [], [], []
    def sync():
        if torch.cuda.is_available() and DEVICE != "cpu":
            torch.cuda.synchronize()

    for (name, seed), weights in BEST.items():
        model = YOLO(str(weights))
        for scope in (["common6", "full8"] if name == "B" else ["common6"]):
            selected = df[(df.split == "test") & df.class_name.isin(BASE if scope == "common6" else NAMES)]
            images = [cv2.imread(str(frames / f"{sample_id(r.video)}.jpg")) for r in selected.itertuples()]
            assert all(im is not None for im in images)
            options = dict(imgsz=IMGSZ, device=DEVICE, conf=CONF, iou=0.7,
                           quantize=32, rect=False, max_det=300, verbose=False)
            for _ in range(10):
                model.predict(images[0], **options)
            totals, extra_count = np.zeros(3, dtype=int), 0
            walls, inference = [], []
            for repeat in range(3):
                for row, image in zip(selected.itertuples(), images):
                    sync()
                    start = time.perf_counter()
                    out = model.predict(image, **options)[0]
                    sync()
                    walls.append((time.perf_counter()-start)*1000)
                    inference.append(out.speed["inference"])
                    if repeat == 0:
                        boxes = out.boxes.xyxy.cpu().numpy()
                        ids = out.boxes.cls.cpu().numpy().astype(int)
                        scores = out.boxes.conf.cpu().numpy()
                        target = np.array([row.x1*row.width, row.y1*row.height, row.x2*row.width, row.y2*row.height], dtype=float)
                        totals += score_one(boxes, ids, scores, target, NAMES.index(row.class_name))
                        extra_count += int((ids >= len(BASE)).sum())
                        predictions.append(dict(model=name, seed=seed, scope=scope, video=row.video,
                            gt_class=row.class_name, boxes=boxes.tolist(), classes=ids.tolist(), scores=scores.tolist()))
            tp, fp, fn = map(int, totals)
            precision, recall = tp/max(tp+fp, 1), tp/max(tp+fn, 1)
            fixed.append(dict(model=name, seed=seed, scope=scope, conf=CONF, match_iou=MATCH_IOU,
                TP=tp, FP=fp, FN=fn, precision_micro=precision, recall_micro=recall,
                F1_micro=2*precision*recall/max(precision+recall, 1e-12),
                predictions_extra_classes=extra_count))
            timing.append(dict(model=name, seed=seed, scope=scope, batch=1, repeats=3,
                inference_mean_ms=float(np.mean(inference)), predict_mean_ms=float(np.mean(walls)),
                predict_p50_ms=float(np.percentile(walls, 50)), predict_p95_ms=float(np.percentile(walls, 95)),
                serial_predict_FPS=1000/float(np.mean(walls))))
        del model
    pd.DataFrame(fixed).to_csv(WORK / "fixed_threshold.csv", index=False)
    pd.DataFrame(timing).to_csv(WORK / "latency.csv", index=False)
    (WORK / "predictions.json").write_text(json.dumps(predictions, indent=2))
    display(pd.DataFrame(fixed))
    display(pd.DataFrame(timing))

In [13]:
if RUN_EVALUATION:
    common = summary[summary.scope == "common6"]
    comparison = common.pivot(index="seed", columns="model", values=["mAP50", "mAP50_95"])
    delta = pd.DataFrame({f"delta_{metric}_pp": 100*(comparison[metric]["B"]-comparison[metric]["A"])
                          for metric in ("mAP50", "mAP50_95")})
    delta.to_csv(WORK / "comparison_common6.csv")
    display(delta)
    if len(SEEDS) > 1:
        display(delta.agg(["mean", "std"]))

## Batas benchmark dan interpretasi
Benchmark batch=1, FP32, 10 warmup, tiga pengulangan pada gambar sama. `inference_mean_ms` adalah tahap inference Ultralytics; `predict_mean_ms`, p50/p95 mencakup preprocessing, inference, postprocessing, dan overhead API. FPS = 1000 / mean wall time predict. Pengukuran tidak mencakup baca file, decode video, MediaPipe, UI, atau kamera. **Bukan uji real-time webcam.**

Satu seed hanya hasil deskriptif. Pengulangan benchmark bukan pengulangan training. Generalisasi eksternal terbatas pada signer/sumber Kaggle yang tersedia. Pengenalan per frame tidak mengukur pemahaman urutan gerakan.